# Session 10: 高度制御 — フィードフォワードとアンチワインドアップ
# Session 10: Altitude Control — Feedforward and Anti-Windup

## 目標 / Objective

ホバー推力フィードフォワードとアンチワインドアップの効果を体感する。

Experience the effect of hover thrust feedforward and anti-windup.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| フィードフォワード | ホバー推力の事前補償 |
| アンチワインドアップ | 積分ワインドアップの防止 |
| 高度制御の特殊性 | 重力補償、地面効果 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from stampfly_edu.sim.cascade import simulate_altitude_control
from stampfly_edu import load_sample_data, plot_step_response

print("Ready! / 準備完了！")

## 1. なぜフィードフォワードが必要か / Why Feedforward?

高度制御の方程式：

$$m \ddot{z} = F_{thrust} - mg$$

ホバーするには $F_{thrust} = mg$ が必要。

フィードフォワードなし → PID の積分項がゆっくり mg に追いつく → 遅い応答  
フィードフォワードあり → 初期推力 = mg、PID は微調整のみ → 速い応答

In [ ]:
# Compare with and without feedforward
# フィードフォワードあり vs なしの比較

result_ff = simulate_altitude_control(
    ff_enabled=True, Kp=2.0, Ti=1.0, Td=0.1,
    setpoint_m=0.5, step_time=2.0, step_size=0.3, duration=8.0,
)
result_no_ff = simulate_altitude_control(
    ff_enabled=False, Kp=2.0, Ti=1.0, Td=0.1,
    setpoint_m=0.5, step_time=2.0, step_size=0.3, duration=8.0,
)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(result_ff['time'], result_ff['altitude'], 'b-', linewidth=1.5, label='With FF')
axes[0].plot(result_no_ff['time'], result_no_ff['altitude'], 'r--', linewidth=1.5, label='No FF')
axes[0].plot(result_ff['time'], result_ff['setpoint'], 'k:', linewidth=0.8, label='Setpoint')
axes[0].set_ylabel('Altitude (m) / 高度')
axes[0].set_title('Feedforward Effect / フィードフォワード効果')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(result_ff['time'], result_ff['thrust']*1000, 'b-', label='With FF')
axes[1].plot(result_no_ff['time'], result_no_ff['thrust']*1000, 'r--', label='No FF')
axes[1].set_ylabel('Thrust (mN) / 推力')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(result_ff['time'], result_ff['pid_component']*1000, 'b-', label='With FF')
axes[2].plot(result_no_ff['time'], result_no_ff['pid_component']*1000, 'r--', label='No FF')
axes[2].set_ylabel('PID component (mN)')
axes[2].set_xlabel('Time (s) / 時間')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. アンチワインドアップ / Anti-Windup

### 問題：積分ワインドアップ

出力が飽和中も積分項が蓄積 → 飽和解除後にオーバーシュートが発生

In [ ]:
# Anti-windup effect
# アンチワインドアップの効果
result_aw = simulate_altitude_control(
    ff_enabled=True, antiwindup=True,
    Kp=5.0, Ti=0.5, Td=0.1,
    setpoint_m=0.3, step_time=1.0, step_size=0.5, duration=8.0,
)
result_no_aw = simulate_altitude_control(
    ff_enabled=True, antiwindup=False,
    Kp=5.0, Ti=0.5, Td=0.1,
    setpoint_m=0.3, step_time=1.0, step_size=0.5, duration=8.0,
)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(result_aw['time'], result_aw['altitude'], 'b-', linewidth=1.5, label='With anti-windup')
axes[0].plot(result_no_aw['time'], result_no_aw['altitude'], 'r--', linewidth=1.5, label='No anti-windup')
axes[0].plot(result_aw['time'], result_aw['setpoint'], 'k:', linewidth=0.8, label='Setpoint')
axes[0].set_ylabel('Altitude (m) / 高度')
axes[0].set_title('Anti-Windup Effect / アンチワインドアップ効果')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(result_aw['time'], result_aw['thrust']*1000, 'b-', label='With AW')
axes[1].plot(result_no_aw['time'], result_no_aw['thrust']*1000, 'r--', label='No AW')
axes[1].set_ylabel('Thrust (mN) / 推力')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. サンプルデータ分析 / Sample Data Analysis

In [ ]:
# Analyze altitude step response from sample data
# サンプルデータの高度ステップ応答を分析
alt_log = load_sample_data("altitude_step")

ax = plot_step_response(
    alt_log, output_col='z', setpoint_col='setpoint',
    title='Altitude Step Response / 高度ステップ応答'
)
plt.show()

## 4. 考察課題 / Discussion Questions

1. **FF の精度**: ホバー推力の推定が 10% ずれた場合、応答にどう影響するか？
2. **バッテリー電圧低下**: 飛行中にバッテリー電圧が下がると FF はどうなるか？
3. **地面効果**: 低高度でのホバリングがなぜ難しいか？

---

1. **FF Accuracy**: How does a 10% error in hover thrust estimate affect response?
2. **Battery Voltage Drop**: What happens to FF as battery voltage decreases?
3. **Ground Effect**: Why is low-altitude hovering more difficult?

In [ ]:
# Your experiments here / ここで実験してみてください
